In [18]:
# ========== 前序遍历实现 ==========
class TreeNode:
    def __init__(self, level, weight, value):
        self.level = level    # 当前决策的物品索引
        self.weight = weight  # 当前总重量
        self.value = value    # 当前总价值
        self.left = None      # 左子树：选择该物品
        self.right = None     # 右子树：不选该物品

class KnapsackTree:
    def __init__(self, items, capacity):
        self.items = items
        self.capacity = capacity
        self.n = len(items)
        self.best_value = 0
        self.best_path = []
        
    def build_and_search(self):
        """构建并搜索决策树"""
        # 创建根节点
        root = TreeNode(-1, 0, 0)
        
        # 前序遍历构建和搜索
        self.preorder_traverse(root, [])
        return self.best_value, self.best_path
    
    def preorder_traverse(self, node, path):
        """前序遍历：节点 → 左 → 右"""
        # 访问当前节点（处理逻辑）
        if node.level == self.n - 1 or node.weight >= self.capacity:
            # 到达叶子节点或超重
            if node.value > self.best_value:
                self.best_value = node.value
                self.best_path = path.copy()
            return
        
        current_level = node.level + 1
        weight, value = self.items[current_level]
        
        # 创建左孩子（选）
        if node.weight + weight <= self.capacity:
            left_child = TreeNode(current_level, 
                                 node.weight + weight,
                                 node.value + value)
            node.left = left_child
            self.preorder_traverse(left_child, path + [1])
        

        right_child = TreeNode(current_level,
                              node.weight,
                              node.value)
        node.right = right_child
        self.preorder_traverse(right_child, path + [0])


In [19]:

# 使用示例
items = [(2, 3), (3, 4), (4, 5), (5, 6)]
solver = KnapsackTree(items, 10)
best_value, path = solver.build_and_search()
print(f"前序遍历结果: 价值={best_value}, 路径={path}")

前序遍历结果: 价值=13, 路径=[1, 1, 0, 1]


In [25]:
class TreeNode:
    def __init__(self, level, weight, value, bound=0):
        self.level = level
        self.weight = weight
        self.value = value
        self.bound = bound      # 上界
        self.left = None
        self.right = None
        # self.x = 0
        # self.y = 0

class KnapsackTree1:
    def __init__(self, items, capacity):
        self.items = items
        self.capacity = capacity
        self.n = len(items)
        self.best_value = 0
        self.best_path = []
        self.nodes = []          # 用于可视化
        self.pruned_edges = []   # 记录被剪掉的边

    # ======== 上界函数（分数背包估计）========
    def bound(self, level, weight, value):
        # 只有“真正超重”才直接剪枝
        if weight > self.capacity:
            return 0

        bound = value
        total_weight = weight
        j = level + 1

        # 尽量贪心装后面的物品
        while j < self.n and total_weight + self.items[j][0] <= self.capacity:
            total_weight += self.items[j][0]
            bound += self.items[j][1]
            j += 1

        # 最后一个物品按比例装入（分数背包）
        if j < self.n:
            w, v = self.items[j]
            bound += (self.capacity - total_weight) * v / w

        return bound

    def build_and_search(self):
        root = TreeNode(-1, 0, 0)
        root.bound = self.bound(-1, 0, 0)
        self.preorder_traverse(root, [])
        return self.best_value, self.best_path

    # ======== 前序遍历 + 分支限界剪枝 ========
    def preorder_traverse(self, node, path):
        self.nodes.append(node)

        # 到达最后一层（已经决定完最后一个物品）
        if node.level == self.n - 1:
            if node.value > self.best_value:
                self.best_value = node.value
                self.best_path = path.copy()
            return

        current_level = node.level + 1
        weight, value = self.items[current_level]

        # -------- 左分支（选该物品）--------
        if node.weight + weight <= self.capacity:
            left = TreeNode(
                current_level,
                node.weight + weight,
                node.value + value
            )
            left.bound = self.bound(current_level, left.weight, left.value)
            node.left = left

            if left.bound > self.best_value:
                self.preorder_traverse(left, path + [1])
            else:
                # 记录被剪枝的边
                self.pruned_edges.append((node, left))

        # -------- 右分支（不选该物品）--------
        right = TreeNode(
            current_level,
            node.weight,
            node.value
        )
        right.bound = self.bound(current_level, right.weight, right.value)
        node.right = right

        if right.bound > self.best_value:
            self.preorder_traverse(right, path + [0])
        else:
            self.pruned_edges.append((node, right))

   



In [21]:

# 使用示例
items = [(2, 3), (3, 4), (4, 5), (5, 6)]
solver = KnapsackTree1(items, 10)
best_value, path = solver.build_and_search()
print(f"前序遍历结果: 价值={best_value}, 路径={path}")

前序遍历结果: 价值=13, 路径=[1, 1, 0, 1]


In [22]:
import random

def generate_items(n, w_range=(1, 20), v_range=(1, 30)):
    return [(random.randint(*w_range), random.randint(*v_range)) for _ in range(n)]

In [23]:
# items = generate_items(20)
# solver = KnapsackTree(items, 100)
# best_value, path = solver.build_and_search()
# print(f"前序遍历结果: 价值={best_value}, 路径={path}")


In [24]:
# solver = KnapsackTree1(items, 100)
# best_value, path = solver.build_and_search()
# print(f"前序遍历结果: 价值={best_value}, 路径={path}")